# 05 — Analysis & Results

**ECE1513 Course Project — Traffic Congestion Prediction near U of T St. George Campus**

This notebook performs an in-depth analysis of the best model's predictions. We examine:

1. Congestion heatmap by hour and day of week.
2. Error analysis across different times of day.
3. Feature importance deep dive.
4. Per-location performance.
5. Congestion-level classification accuracy.
6. Travel time estimation examples.
7. Summary of findings and conclusions.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, accuracy_score
)

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load Best Model and Test Data

In [ ]:
# Load the best model (XGBoost from Notebook 04)
best_model = joblib.load('../results/models/xgboost.pkl')
imputer = joblib.load('../results/models/imputer.pkl')

with open('../results/models/feature_cols.json', 'r') as f:
    feature_cols = json.load(f)

test_df = pd.read_csv('../data/processed/test.csv')
train_df = pd.read_csv('../data/processed/train.csv')

TARGET = 'avg_speed'
test_df['time_start'] = pd.to_datetime(test_df['time_start'])

X_test = imputer.transform(test_df[feature_cols].values)
y_test = test_df[TARGET].values
y_pred = best_model.predict(X_test)

test_df['predicted_speed'] = y_pred
test_df['residual'] = y_test - y_pred

print(f'Test set size: {len(test_df):,}')
print(f'MAE  = {mean_absolute_error(y_test, y_pred):.3f} km/h')
print(f'RMSE = {np.sqrt(mean_squared_error(y_test, y_pred)):.3f} km/h')
print(f'R2   = {r2_score(y_test, y_pred):.4f}')

## 2. Congestion Heatmap by Hour and Day of Week

Visualize average actual speeds across the week to identify when congestion is most severe on city streets near U of T.

In [ ]:
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# Actual speeds heatmap
heatmap_actual = test_df.pivot_table(
    values=TARGET, index='hour_of_day', columns='day_of_week', aggfunc='mean'
)
heatmap_actual.columns = [day_labels[i] for i in heatmap_actual.columns]

# Predicted speeds heatmap
heatmap_pred = test_df.pivot_table(
    values='predicted_speed', index='hour_of_day', columns='day_of_week', aggfunc='mean'
)
heatmap_pred.columns = [day_labels[i] for i in heatmap_pred.columns]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

sns.heatmap(heatmap_actual, annot=True, fmt='.0f', cmap='RdYlGn', linewidths=0.5,
            cbar_kws={'label': 'Avg Speed (km/h)'}, ax=axes[0], vmin=15, vmax=60)
axes[0].set_title('Actual Average Speed')
axes[0].set_ylabel('Hour of Day')
axes[0].set_xlabel('Day of Week')

sns.heatmap(heatmap_pred, annot=True, fmt='.0f', cmap='RdYlGn', linewidths=0.5,
            cbar_kws={'label': 'Avg Speed (km/h)'}, ax=axes[1], vmin=15, vmax=60)
axes[1].set_title('Predicted Average Speed (XGBoost)')
axes[1].set_ylabel('Hour of Day')
axes[1].set_xlabel('Day of Week')

plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/congestion_heatmap.png', bbox_inches='tight')
plt.show()

## 3. Error Analysis by Time of Day

Examine how prediction errors vary throughout the day. Larger errors during rush hours could indicate the model struggles with extreme congestion events.

In [ ]:
error_by_hour = test_df.groupby('hour_of_day').agg(
    MAE=('residual', lambda x: np.mean(np.abs(x))),
    Mean_Residual=('residual', 'mean'),
    Std_Residual=('residual', 'std'),
    Count=('residual', 'count')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(error_by_hour['hour_of_day'], error_by_hour['MAE'], color='steelblue', edgecolor='white')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('MAE (km/h)')
axes[0].set_title('Mean Absolute Error by Hour')
axes[0].set_xticks(range(24))

axes[1].bar(error_by_hour['hour_of_day'], error_by_hour['Mean_Residual'], color='coral', edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Mean Residual (km/h)')
axes[1].set_title('Mean Residual (Bias) by Hour')
axes[1].set_xticks(range(24))

plt.tight_layout()
plt.savefig('../results/figures/error_by_hour.png', bbox_inches='tight')
plt.show()

In [ ]:
# Error by weekend vs weekday
for label, group in test_df.groupby('is_weekend'):
    subset_mae = mean_absolute_error(group[TARGET], group['predicted_speed'])
    subset_rmse = np.sqrt(mean_squared_error(group[TARGET], group['predicted_speed']))
    name = 'Weekend' if label == 1 else 'Weekday'
    print(f'{name:10s}  MAE={subset_mae:.3f}  RMSE={subset_rmse:.3f}  n={len(group):,}')

## 4. Feature Importance Deep Dive

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 features
importances.head(15).plot(kind='barh', ax=axes[0], color='#e74c3c')
axes[0].invert_yaxis()
axes[0].set_title('XGBoost — Top 15 Feature Importances')
axes[0].set_xlabel('Importance (gain)')

# Cumulative importance
cum_imp = importances.cumsum() / importances.sum()
axes[1].plot(range(len(cum_imp)), cum_imp.values, marker='o', markersize=3)
axes[1].axhline(0.9, color='red', linestyle='--', label='90% threshold')
n_90 = (cum_imp <= 0.9).sum()
axes[1].axvline(n_90, color='gray', linestyle=':')
axes[1].set_xlabel('Feature rank')
axes[1].set_ylabel('Cumulative importance')
axes[1].set_title(f'Cumulative Feature Importance ({n_90} features for 90%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/feature_importance_deep_dive.png', bbox_inches='tight')
plt.show()

## 5. Per-Location Performance

Examine how model performance varies across different street locations near U of T.

In [ ]:
# Per-location MAE
loc_perf = test_df.groupby('location_name').apply(
    lambda g: pd.Series({
        'MAE': mean_absolute_error(g[TARGET], g['predicted_speed']),
        'RMSE': np.sqrt(mean_squared_error(g[TARGET], g['predicted_speed'])),
        'Mean_Speed': g[TARGET].mean(),
        'Count': len(g)
    })
).sort_values('MAE', ascending=False)

print(f'Per-location performance ({len(loc_perf)} locations):')
print(f'  Best MAE:  {loc_perf["MAE"].min():.3f} km/h at {loc_perf["MAE"].idxmin()}')
print(f'  Worst MAE: {loc_perf["MAE"].max():.3f} km/h at {loc_perf["MAE"].idxmax()}')
print(f'  Median MAE: {loc_perf["MAE"].median():.3f} km/h')

loc_perf.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of per-location MAE
axes[0].hist(loc_perf['MAE'], bins=30, edgecolor='white', color='steelblue')
axes[0].set_xlabel('MAE (km/h)')
axes[0].set_ylabel('Number of Locations')
axes[0].set_title('Distribution of Per-Location MAE')
axes[0].axvline(loc_perf['MAE'].median(), color='red', linestyle='--',
                label=f'Median: {loc_perf["MAE"].median():.1f}')
axes[0].legend()

# MAE vs mean speed (locations with higher mean speeds may have different error patterns)
axes[1].scatter(loc_perf['Mean_Speed'], loc_perf['MAE'], alpha=0.6, s=20)
axes[1].set_xlabel('Mean Speed at Location (km/h)')
axes[1].set_ylabel('MAE (km/h)')
axes[1].set_title('Per-Location MAE vs Mean Speed')

plt.tight_layout()
plt.savefig('../results/figures/per_location_performance.png', bbox_inches='tight')
plt.show()

In [ ]:
# Time series: actual vs predicted for a few sample locations
top_locations = test_df['location_name'].value_counts().head(4).index.tolist()
n_plots = len(top_locations)

fig, axes = plt.subplots(n_plots, 1, figsize=(14, 4 * n_plots), sharex=False)
if n_plots == 1:
    axes = [axes]

for i, loc in enumerate(top_locations):
    loc_data = test_df[test_df['location_name'] == loc].sort_values('time_start').head(300)
    axes[i].plot(loc_data['time_start'], loc_data[TARGET], label='Actual', linewidth=0.8)
    axes[i].plot(loc_data['time_start'], loc_data['predicted_speed'], label='Predicted', linewidth=0.8, alpha=0.8)
    axes[i].set_title(f'{loc}')
    axes[i].set_ylabel('Speed (km/h)')
    axes[i].legend(loc='upper right')

axes[-1].set_xlabel('Date / Time')
plt.tight_layout()
plt.savefig('../results/figures/location_predictions.png', bbox_inches='tight')
plt.show()

## 6. Congestion Level Classification Accuracy

Although we train a regression model to predict speed, we can convert predicted speeds into congestion levels and evaluate classification performance. The city-street thresholds are: gridlock (< 20), heavy (20-35), moderate (35-50), free_flow (>= 50 km/h).

In [ ]:
# Define congestion bins for city streets
bins = [-np.inf, 20, 35, 50, np.inf]
label_names = ['gridlock', 'heavy', 'moderate', 'free_flow']
label_codes = [3, 2, 1, 0]

# Actual congestion levels
y_cls_true = pd.cut(test_df[TARGET], bins=bins, labels=label_codes, right=False).astype(int).values

# Predicted congestion levels
y_cls_pred = pd.cut(pd.Series(y_pred), bins=bins, labels=label_codes, right=False).astype(int).values

acc = accuracy_score(y_cls_true, y_cls_pred)
print(f'Congestion Level Classification Accuracy: {acc:.4f}\n')

print(classification_report(
    y_cls_true, y_cls_pred,
    target_names=['Free Flow (0)', 'Moderate (1)', 'Heavy (2)', 'Gridlock (3)']
))

In [ ]:
cm = confusion_matrix(y_cls_true, y_cls_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Free', 'Moderate', 'Heavy', 'Gridlock'],
    yticklabels=['Free', 'Moderate', 'Heavy', 'Gridlock'],
    ax=ax
)
ax.set_xlabel('Predicted Level')
ax.set_ylabel('Actual Level')
ax.set_title('Congestion Level Confusion Matrix')
plt.tight_layout()
plt.savefig('../results/figures/congestion_confusion_matrix.png', bbox_inches='tight')
plt.show()

## 7. Travel Time Estimation

As a practical application, we estimate travel times for a typical city block (~500 m) using predicted speeds and compare with actual travel times derived from observed speeds.

In [ ]:
# Assume a representative city block length of 0.5 km
BLOCK_LENGTH_KM = 0.5

test_df['actual_travel_time_sec'] = (BLOCK_LENGTH_KM / test_df[TARGET].clip(lower=1)) * 3600
test_df['predicted_travel_time_sec'] = (BLOCK_LENGTH_KM / test_df['predicted_speed'].clip(lower=1)) * 3600

# Show a few example rows
example_cols = ['time_start', 'location_name', TARGET, 'predicted_speed',
                'actual_travel_time_sec', 'predicted_travel_time_sec']
print('Travel time estimation examples (500 m block):')
test_df[example_cols].sample(10, random_state=42).round(1)

In [ ]:
tt_mae = mean_absolute_error(
    test_df['actual_travel_time_sec'].dropna(),
    test_df['predicted_travel_time_sec'].loc[test_df['actual_travel_time_sec'].dropna().index]
)
print(f'Travel Time MAE: {tt_mae:.1f} seconds (for a 500 m block)')

fig, ax = plt.subplots(figsize=(10, 5))
hourly_tt = test_df.groupby('hour_of_day')[['actual_travel_time_sec', 'predicted_travel_time_sec']].mean()
hourly_tt.plot(kind='bar', ax=ax, color=['steelblue', 'coral'], edgecolor='white')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Travel Time (seconds per 500 m)')
ax.set_title('Average Travel Time by Hour — Actual vs Predicted')
ax.legend(['Actual', 'Predicted'])
plt.tight_layout()
plt.savefig('../results/figures/travel_time_by_hour.png', bbox_inches='tight')
plt.show()

## 8. Summary of Findings and Conclusions

### Key Results

- **XGBoost** is the best-performing model, achieving the lowest MAE and highest R-squared on the held-out test set. It outperforms the Historical Average baseline by a meaningful margin.
- **Temporal features** (hour of day, day of week) and **location features** (location_id, direction) are the most important predictors for city street speeds near U of T. This reflects the fact that urban traffic is heavily driven by commuting patterns and road-specific characteristics.
- **Weather variables** (temperature, precipitation, visibility) contribute to predictions but are less dominant than on highways, likely because city street speeds are already constrained by traffic signals and congestion.
- **Error analysis** reveals that prediction errors are somewhat larger during rush-hour transitions (7-9 AM, 4-6 PM), where traffic conditions change rapidly. Weekend predictions tend to be slightly more accurate due to less variable conditions.
- The model achieves strong **congestion classification accuracy**, correctly identifying the congestion level in the majority of test cases. Misclassifications tend to occur at level boundaries (e.g., heavy vs. moderate).
- **Travel time estimates** derived from predicted speeds are practically useful for short city blocks, with small per-block errors that would compound over longer routes.

### Limitations

- The model does not account for non-recurrent congestion events (accidents, construction, special events near campus).
- Weather data is from Toronto Pearson airport, which may not perfectly represent microclimatic conditions downtown.
- The temporal train/test split means the model has not seen the exact dates in the test set, though it has seen the same hours and days of week.
- Speed-bin data provides 15-minute aggregates, which may miss very short-duration congestion spikes.

### Future Work

- Incorporate real-time incident data from the City of Toronto.
- Experiment with deep learning approaches (LSTM, Transformer) that can better model sequential dependencies.
- Extend the analysis to other neighbourhoods in Toronto.
- Add U of T academic calendar features (e.g., exam periods, reading week) which may affect traffic patterns near campus.
- Deploy a lightweight API for real-time congestion prediction.